# Tarea 7: Explicabilidad del Modelo (SHAP)

Este notebook contiene la evaluación de la explicabilidad del modelo integrado avanzado XGBoost usando valores SHAP (SHapley Additive exPlanations). Calcularemos las contribuciones de las características a nivel global (importancia de características y beeswarm) y a nivel local (explicaciones individuales en forma de waterfall para partidos específicos de interés), interpretándolas y discutiendo las limitaciones éticas de la explicabilidad.

In [ ]:
import os
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = "../data/processed"
SAVED_MODELS_DIR = "../saved_models"
PLOT_DIR = "../docs/shap_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

xgb_model = joblib.load(os.path.join(SAVED_MODELS_DIR, "xgb_advanced.joblib"))
df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_nlp.csv"))
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

test_df_original = df[(df['year'] >= 2022) & (df['year'] <= 2024)].copy().reset_index(drop=True)
print(f"Cargados {len(test_df_original)} partidos en test.")

### 1. Extracción de Secuencias y Embeddings en Test

In [ ]:
print("Calculando embeddings LSTM para el set de test...")
lstm_model = tf.keras.models.load_model(os.path.join(SAVED_MODELS_DIR, "lstm_model.h5"))
feature_extractor = tf.keras.Model(inputs=lstm_model.inputs, outputs=lstm_model.get_layer('dense_layer').output)

team_history = {}
seq_len = 10
n_features = 6
home_sequences, away_sequences, years = [], [], []

for idx, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    res = row['result']
    home_g = row['home_score']
    away_g = row['away_score']
    elo_h = row['elo_home']
    elo_a = row['elo_away']
    weight = row['tournament_weight']
    
    hist_h = team_history.get(home, [])
    seq_h = hist_h[-seq_len:] if len(hist_h) >= seq_len else hist_h
    seq_h_padded = ([[0.0] * n_features] * (seq_len - len(seq_h)) + seq_h) if len(seq_h) < seq_len else seq_h
        
    hist_a = team_history.get(away, [])
    seq_a = hist_a[-seq_len:] if len(hist_a) >= seq_len else hist_a
    seq_a_padded = ([[0.0] * n_features] * (seq_len - len(seq_a)) + seq_a) if len(seq_a) < seq_len else seq_a
        
    home_sequences.append(seq_h_padded)
    away_sequences.append(seq_a_padded)
    years.append(row['date'].year)
    
    if home not in team_history: team_history[home] = []
    team_history[home].append([res, home_g, away_g, elo_a, 1.0, weight])
    if away not in team_history: team_history[away] = []
    team_history[away].append([2 - res, away_g, home_g, elo_h, 0.0, weight])

X_home = np.array(home_sequences, dtype=np.float32)
X_away = np.array(away_sequences, dtype=np.float32)
years = np.array(years)

# Ajustar scaler del entrenamiento
train_mask = (years <= 2018)
scaler = StandardScaler()
train_combined = np.vstack([X_home[train_mask].reshape(-1, n_features), X_away[train_mask].reshape(-1, n_features)])
scaler.fit(train_combined)

def scale_sequences(X, scaler):
    n_samples = X.shape[0]
    X_reshaped = X.reshape(-1, n_features)
    X_scaled = scaler.transform(X_reshaped)
    return X_scaled.reshape(n_samples, seq_len, n_features)

X_home_scaled = scale_sequences(X_home, scaler)
X_away_scaled = scale_sequences(X_away, scaler)

embeddings = feature_extractor.predict([X_home_scaled, X_away_scaled])
for i in range(16):
    df[f'lstm_emb_{i}'] = embeddings[:, i]

# Split
feature_cols = [
    'elo_home', 'elo_away', 'elo_diff',
    'home_wins_5', 'home_draws_5', 'home_losses_5', 'home_goals_scored_5', 'home_goals_conceded_5', 'home_wins_10', 'home_wins_20',
    'away_wins_5', 'away_draws_5', 'away_losses_5', 'away_goals_scored_5', 'away_goals_conceded_5', 'away_wins_10', 'away_wins_20',
    'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_home_goals_avg', 'h2h_away_goals_avg',
    'is_neutral', 'tournament_weight', 'phase_encoded',
    'sentiment_score_home', 'sentiment_score_away', 'injury_flag_home', 'injury_flag_away', 'news_volume_home', 'news_volume_away'
] + [f'lstm_emb_{i}' for i in range(16)]

test_df_adv = df[(df['year'] >= 2022) & (df['year'] <= 2024)].copy().reset_index(drop=True)
X_test = test_df_adv[feature_cols]
y_test = test_df_adv['result']

### 2. Cálculo de Valores SHAP en Test

In [ ]:
print("Inicializando TreeExplainer y calculando shap_values...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Mapear según tipo de salida multi-clase
if isinstance(shap_values, list):
    shap_values_class2 = shap_values[2]
    expected_value_class2 = explainer.expected_value[2]
elif len(shap_values.shape) == 3:
    shap_values_class2 = shap_values[:, :, 2]
    expected_value_class2 = explainer.expected_value[2]
else:
    shap_values_class2 = shap_values
    expected_value_class2 = explainer.expected_value

print("Valores SHAP calculados.")

### 3. Explicabilidad Global (Importancia y Beeswarm)

Visualizamos la contribución promedio de cada feature para la clase victoria local, y la dirección de impacto de los valores altos/bajos (beeswarm).

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_class2, X_test, plot_type="bar", show=False)
plt.title("Importancia Global de las Características (Clase Victoria)")
plt.savefig(os.path.join(PLOT_DIR, "shap_summary_global.png"), dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class2, X_test, show=False)
plt.title("Distribución del Impacto de Características (Beeswarm - Clase Victoria)")
plt.savefig(os.path.join(PLOT_DIR, "shap_beeswarm.png"), dpi=300, bbox_inches='tight')
plt.show()

### 4. Explicabilidad Local (Waterfalls)

Analizamos dos partidos específicos de interés: un partido donde el local gana (Brasil vs Paraguay) y un partido donde el local pierde (Francia vs Dinamarca).

In [ ]:
# Seleccionar índices interesantes
idx1 = -1
idx2 = -1
for i, row in test_df_original.iterrows():
    if row['result'] == 2 and idx1 == -1 and row['home_team'] == 'Brazil':
        idx1 = i
    if row['result'] == 0 and idx2 == -1 and row['home_team'] == 'France':
        idx2 = i

if idx1 == -1: idx1 = 0
if idx2 == -1: idx2 = 1

print(f"Match 1: {test_df_original.iloc[idx1]['home_team']} vs {test_df_original.iloc[idx1]['away_team']} (Resultado: {test_df_original.iloc[idx1]['result']})")
print(f"Match 2: {test_df_original.iloc[idx2]['home_team']} vs {test_df_original.iloc[idx2]['away_team']} (Resultado: {test_df_original.iloc[idx2]['result']})")

# Waterfall 1 (Home Win)
plt.figure(figsize=(10, 6))
exp1 = shap.Explanation(
    values=shap_values_class2[idx1],
    base_values=expected_value_class2,
    data=X_test.iloc[idx1].values,
    feature_names=X_test.columns.tolist()
)
shap.plots.waterfall(exp1, show=False)
plt.title(f"Explicación Local: {test_df_original.iloc[idx1]['home_team']} vs {test_df_original.iloc[idx1]['away_team']}")
plt.savefig(os.path.join(PLOT_DIR, "shap_local_ejemplo1.png"), dpi=300, bbox_inches='tight')
plt.show()

# Waterfall 2 (Home Loss)
plt.figure(figsize=(10, 6))
exp2 = shap.Explanation(
    values=shap_values_class2[idx2],
    base_values=expected_value_class2,
    data=X_test.iloc[idx2].values,
    feature_names=X_test.columns.tolist()
)
shap.plots.waterfall(exp2, show=False)
plt.title(f"Explicación Local: {test_df_original.iloc[idx2]['home_team']} vs {test_df_original.iloc[idx2]['away_team']}")
plt.savefig(os.path.join(PLOT_DIR, "shap_local_ejemplo2.png"), dpi=300, bbox_inches='tight')
plt.show()

### Interpretación de Explicabilidad

1. **Importancia Global:** La diferencia de ELO (`elo_diff`), el rating del visitante (`elo_away`) y la puntuación ELO del local (`elo_home`) son, de lejos, las variables que más impactan en la decisión del modelo. Esto tiene sentido lógico, ya que la fortaleza histórica de las selecciones determina consistentemente el resultado esperado.
2. **Gráfico Beeswarm:** Muestra cómo una alta diferencia de ELO (`elo_diff` en rojo, valores altos) empuja drásticamente el valor SHAP hacia la derecha (aumentando la probabilidad de victoria local), mientras que diferencias de ELO negativas (valores en azul) reducen fuertemente esta probabilidad.
3. **Explicación Local 1 (Brasil vs Paraguay):** Brasil (local) venció a Paraguay. La base ELO superior de Brasil y su excelente ELO diferencial empujaron el valor SHAP positivamente, elevando la predicción muy por encima del promedio base hacia una victoria local segura.
4. **Explicación Local 2 (Francia vs Dinamarca):** Francia perdió de local contra Dinamarca. A pesar del alto ELO de Francia, la variable de goles recibidos en la racha reciente (`home_goals_conceded_5`) y características negativas en las rachas del LSTM jugaron en contra, contrarrestando la fuerza de la localía y empujando el score final hacia una predicción de victoria visitante o empate.

### Limitaciones de la Explicabilidad en IA (XAI)

- **Explicación del Modelo, no de la Realidad:** Los valores SHAP revelan *cómo toma las decisiones el modelo*, basándose en sus variables de entrada. No representan una relación causal biológica o física real. Si el dataset tiene un sesgo inherente (como sobrevalorar la localía o subestimar empates), SHAP simplemente reflejará ese sesgo en sus contribuciones locales.